In [9]:
import config

In [5]:
import json
from pathlib import Path


fasta_path = Path(f"{config.DIR_REPBASE_PROCESSED}/all_sequences_filtered_02_ltr_correction.fasta")
meta_path = Path(f"{config.DIR_REPBASE_PROCESSED}/metadata_02_ltr_correction.json")
hierarchy_path = Path(f"{config.DIR_REPBASE_PROCESSED}/hierarchy_sequences_02_ltr_correction.json")


def read_fasta(path):
    records = []
    header = None
    seq_parts = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if line.startswith(">"):
                if header is not None:
                    records.append((header, "".join(seq_parts)))
                header = line[1:]
                seq_parts = []
            else:
                seq_parts.append(line.strip())

        if header is not None:
            records.append((header, "".join(seq_parts)))

    return records


def append_fasta(path, records, width=80):
    with open(path, "a", encoding="utf-8") as f:
        if records:
            f.write("\n")
        for header, seq in records:
            f.write(f">{header}\n")
            for i in range(0, len(seq), width):
                f.write(seq[i:i + width] + "\n")


def load_json(path):
    if not path.exists():
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def remove_marker(seq_id):
    if "-LTR" in seq_id:
        return seq_id.replace("-LTR", ""), "LTR"
    if "-I" in seq_id:
        return seq_id.replace("-I", ""), "I"
    return None, None


def add_seq_id(node, seq_id):
    if "sequences" not in node or node["sequences"] is None:
        node["sequences"] = []
    if seq_id not in node["sequences"]:
        node["sequences"].append(seq_id)


def add_seq_id_up_tree(tree, target_class, seq_id):
    def dfs(node_dict):
        for class_name, class_node in node_dict.items():
            subs = class_node.get("subs", {})

            if class_name == target_class:
                add_seq_id(class_node, seq_id)
                return True

            if isinstance(subs, dict) and dfs(subs):
                add_seq_id(class_node, seq_id)
                return True

        return False

    return dfs(tree)


# загрузка
fasta_records = read_fasta(fasta_path)
meta = load_json(meta_path)
hierarchy = load_json(hierarchy_path)

existing_fasta_ids = set()
parsed = []

for seq_id, seq in fasta_records:
    existing_fasta_ids.add(seq_id)
    base_id, kind = remove_marker(seq_id)
    parsed.append({
        "seq_id": seq_id,
        "seq": seq,
        "base_id": base_id,
        "kind": kind,
    })

# группировка пар
pairs = {}
for rec in parsed:
    if rec["base_id"] is None:
        continue
    pairs.setdefault(rec["base_id"], {})
    pairs[rec["base_id"]][rec["kind"]] = rec


new_fasta_records = []
new_ids = []  # только реально новые base_id
added_seq_count = 0
added_meta_count = 0
added_hierarchy_count = 0

for base_id, kinds in pairs.items():
    if "LTR" not in kinds or "I" not in kinds:
        continue

    # если уже был вообще пропускаем
    if base_id in existing_fasta_ids:
        continue

    ltr_id = kinds["LTR"]["seq_id"]
    i_id = kinds["I"]["seq_id"]

    # FASTA
    ltr_seq = kinds["LTR"]["seq"]
    i_seq = kinds["I"]["seq"]
    new_seq = ltr_seq + i_seq + ltr_seq

    new_fasta_records.append((base_id, new_seq))
    existing_fasta_ids.add(base_id)
    new_ids.append(base_id)
    added_seq_count += 1

    # meta.json
    if base_id not in meta:
        if ltr_id in meta:
            meta[base_id] = dict(meta[ltr_id])
            added_meta_count += 1
        elif i_id in meta:
            meta[base_id] = dict(meta[i_id])
            added_meta_count += 1

# записываем FASTA
if new_fasta_records:
    append_fasta(fasta_path, new_fasta_records)

# обновляем hierarchy только для реально новых
for base_id in new_ids:
    if base_id in meta and isinstance(meta[base_id], dict):
        class_name = meta[base_id].get("class")
        if class_name:
            found = add_seq_id_up_tree(hierarchy, class_name, base_id)
            if found:
                added_hierarchy_count += 1
            else:
                print(f'Класс "{class_name}" для {base_id} не найден в hierarchy')

# сохраняем json
save_json(meta_path, meta)
save_json(hierarchy_path, hierarchy)

print(f"Добавлено новых последовательностей в FASTA: {added_seq_count}")
print(f"Добавлено новых записей в meta.json: {added_meta_count}")
print(f"Добавлено новых ID в hierarchy: {added_hierarchy_count}")

Добавлено новых последовательностей в FASTA: 24276
Добавлено новых записей в meta.json: 24276
Добавлено новых ID в hierarchy: 24276


In [6]:
new_fasta_records

[('GYPSY1_CB',
  'tgttgcagactatgcaaaccttcattagttttataagtttaatatccgccttagtttgatctacccaaagtaggcgcgcgcaataaccactgtttgtgctaattgtcatcacgcgcgtaactatttgttcccttggtcagaactcctagataaaggggctgatcattctgatactctctctcctgccccaaagtattctgttgtcccctattcactgaataaagtccttattcatctactactcacatactgtctggctggtggttcacacaatgccgccaccaggagacacaacaactggcgatcaggctacggaattggcggatctcacgaaacaaatcggtttgctagtcaacatcttcgcaaaactcgctcaagcatcggcaaacgctccgttgacaactacacccaccagtggacacaatcttctagtggaatcgatatcaaaacggattccaatcttcacctatgatccagacgacgaccaaaccttcgacacgtggtacgctcgatatgaggatgtcctgactaaggatggtgaatctctagaagaaggtgagaaatcaagggtaattctgtccaaacttagctcgaaagagtactctcacttcacaaatcgtatcctaccgaaactcccgaacgagctcaactttgcggagctcatcagcaaacttcgagagacattcaaatcgacatcatcgattttccggaaacgccaagatttcctccgcacggaatattttggaggagcaattgaagagtatactggacaggtcctcagaaaattcacatcgtccgaattcaagaagatgactgatgatcaagtatgctgtatggtctggataaatggattacgagataacacttactccgacatccgaacaaaggctctccaagttatggaagcgaaaccagaatgtacactgttagaattggagcagaatatcaaacggcttctggatgtgcgagctgactccaaaag

In [7]:
def write_fasta(path, records, width=80):
    with open(path, "w", encoding="utf-8") as f:
        for header, seq in records:
            f.write(f">{header}\n")
            for i in range(0, len(seq), width):
                f.write(seq[i:i + width] + "\n")

In [10]:
output_path = Path(f"{config.DIR_REPBASE_PROCESSED}/all_sequences_filtered_02_only_ltr.fasta")

if new_fasta_records:
    write_fasta(output_path, new_fasta_records)
    print(f"Новые последовательности записаны в {output_path}")
else:
    print("Новых последовательностей нет")

Новые последовательности записаны в /Users/nad/hse/semester08/mobiraph/data/n13_repbase_processed/all_sequences_filtered_02_only_ltr.fasta
